# Amparo -- M3: RAG avanzado (S08) + tool use (S10)

Monta las dos tecnicas avanzadas sobre el RAG ingenuo de S07 y expone el
retrieval como herramienta (function calling). Toda la logica vive en
`tools/rag/` -- aca solo se clona el repo, se instala el stack pesado y se
corren las fases, igual que en `rag_ingenuo.ipynb`.

**Las tres configuraciones que se comparan** (lo UNICO que cambia entre ellas es
el retrieval; mismo prompt, mismo generador, mismo eval set):

| Sistema | `use_hybrid` | `use_rerank` | Que hace |
|---|---|---|---|
| **A** ingenuo (S07) | False | False | denso puro (coseno e5) |
| **B** +hybrid | True | False | denso + BM25, fusion RRF |
| **C** +reranker | True | True | hybrid -> cross-encoder reordena |

**Requiere GPU** (T4 o superior): e5, el cross-encoder y Qwen2.5-7B corren aca.

Las decisiones de diseno de cada tecnica estan en
`docs/m3_decisiones_rag.md`.

> **Nota sobre los deltas.** Este notebook deja LISTA la corrida A/B/C con su
> latencia. La evaluacion en si (scorecard del harness de M2 + las cuatro
> metricas de RAGAS) la corre el companero de evaluacion sobre los registros que
> produce la Fase 4, usando el contrato `pipeline.to_eval_record()`. Los flags
> `used_hybrid`/`used_rerank` de cada registro etiquetan que sistema lo genero.

In [ ]:
# Rama del repo a clonar. Cambiala si el codigo del RAG avanzado todavia no esta
# en main (por ejemplo, mientras vive en una rama de PR).
RAMA = "main"

import os

if not os.path.isdir("Amparo"):
    !git clone -b {RAMA} https://github.com/TomasPosada0626/Amparo.git
else:
    !cd Amparo && git fetch origin && git checkout {RAMA} && git pull

%cd Amparo

In [ ]:
import os, sys
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

REQUERIDOS = [
    "tools/rag/hybrid.py",
    "tools/rag/rerank.py",
    "tools/rag/tools.py",
    "tools/rag/pipeline.py",
    "data/eval_set.json",
]
faltan = [r for r in REQUERIDOS if not os.path.exists(r)]
if faltan:
    raise SystemExit(
        f"Falta en la rama '{RAMA}': {faltan}\n\n"
        "Este notebook corre contra el repo REMOTO. Si el codigo del RAG avanzado "
        "todavia esta solo en tu maquina, commitealo y pusheralo, o cambia RAMA."
    )
print("Repo OK.")

In [ ]:
# Dependencias livianas del repo (incluye faiss-cpu y rank_bm25, el BM25 de la
# hybrid search).
!pip install -q -r requirements.txt

In [ ]:
# Stack pesado -- igual que en M1/M2 y en rag_ingenuo.ipynb: se instala aqui y NO
# en requirements.txt. sentence-transformers trae el cross-encoder del reranking;
# transformers alcanza para e5 y para la generacion; peft/bitsandbytes solo si se
# genera con el adaptador LoRA de M1.
!pip install -q -U transformers sentence-transformers peft bitsandbytes accelerate

## Fase 1 -- Cargar el indice y construir el BM25

El indice denso (FAISS) se construye en `rag_ingenuo.ipynb` y se guarda en Drive.
Aca se carga ya hecho. El indice BM25 se construye en memoria a partir de la
metadata del store (los mismos chunks, en el mismo orden) -- es Python puro y
tarda segundos; se construye UNA vez y se reusa en toda la corrida.

In [ ]:
from google.colab import drive
import pathlib, shutil
from tools.rag import config

drive.mount("/content/drive")
origen = pathlib.Path(config.DRIVE_ROOT) / "rag"
config.ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
for nombre in ("rag_index.faiss", "rag_index_metadata.jsonl"):
    shutil.copy(origen / nombre, config.ARTIFACTS_DIR / nombre)
print("Indice copiado de Drive a artifacts/.")

In [ ]:
from tools.rag import pipeline
from tools.rag.hybrid import BM25Index

store = pipeline.load_index()
bm25 = BM25Index(store.metadata)   # una sola vez; se reusa en cada consulta
print(f"Indice denso: {len(store)} chunks | BM25: {len(bm25)} chunks")

## Fase 2 -- Demo lado a lado: A vs B vs C sobre el RETRIEVAL

Antes de generar nada, se mira SOLO lo que recupera cada sistema. Tres consultas
elegidas para exponer donde cada tecnica paga (o no):

1. **Termino exacto** (`articulo 64` del CST): donde el denso difumina y BM25
   deberia rescatar el chunk correcto -> se espera que B/C lo suban.
2. **Coloquial** (lenguaje del usuario, sin terminos tecnicos): donde el denso
   ya es fuerte -> se espera que hybrid NO lo empeore.
3. **Fuera del corpus**: la valvula de escape debe activarse en los tres.

Solo se recupera (rapido, sin generar): es el diagnostico por consulta que, segun
S08, vale mas que la tabla agregada.

In [ ]:
from tools.rag.retrieve import retrieve

SISTEMAS = {
    "A ingenuo": dict(use_hybrid=False, use_rerank=False),
    "B +hybrid": dict(use_hybrid=True, use_rerank=False),
    "C +rerank": dict(use_hybrid=True, use_rerank=True),
}

consultas_demo = [
    "que dice el articulo 64 del codigo sustantivo del trabajo",   # termino exacto
    "me despidieron sin pagarme la liquidacion, que hago",          # coloquial
    "cual es el plazo para apelar una multa de transito en Argentina",  # fuera de corpus
]

for q in consultas_demo:
    print("=" * 78)
    print("CONSULTA:", q)
    for nombre, banderas in SISTEMAS.items():
        # min_score=None para VER que recupera cada uno antes de la valvula;
        # abajo se muestra aparte si la valvula lo dejaria pasar.
        crudos = retrieve(q, store, top_k=3, min_score=None, bm25=bm25, **banderas)
        citas = [f"{r.cita} (dense={r.dense_score:.2f})" if r.dense_score is not None else f"{r.cita} (solo BM25)" for r in crudos]
        print(f"  {nombre}: {citas}")

## Fase 3 -- Cargar el generador

Qwen2.5-7B-Instruct, el mismo de M1/M2/S07, cargado una sola vez y reusado.
`USE_LORA=True` genera con el adaptador de M1 encima.

In [ ]:
USE_LORA = False
model_bundle = pipeline.load_model(use_lora=USE_LORA)
print("Modelo cargado.")

## Fase 4 -- Corrida A/B/C sobre el eval set (con latencia)

Corre los tres sistemas sobre `data/eval_set.json` y produce, por cada sistema,
la lista de registros en formato Ragas (`pipeline.to_eval_record`). Mide la
**latencia por consulta** de cada sistema -- toda tecnica cobra, y S08 pide
reportar el costo junto al delta.

**Los registros que salen de aca son el insumo del companero de evaluacion**: el
harness de M2 y RAGAS se corren sobre ellos. No se evaluan aca para no duplicar
ese trabajo; lo que este notebook garantiza es que las tres corridas existen,
estan etiquetadas por sistema y son comparables (mismo eval set, misma semilla).

In [ ]:
import json, time
from tools.evaluation import eval_set as eval_set_mod

registros = eval_set_mod.load_eval_set()
print(f"{len(registros)} registros ({len(eval_set_mod.gold_examples(registros))} gold, "
      f"{len(eval_set_mod.adversarial_examples(registros))} adversariales)")

corridas = {}   # nombre_sistema -> lista de eval_records
latencias = {}  # nombre_sistema -> seg/consulta

for nombre, banderas in SISTEMAS.items():
    print(f"\nCorriendo sistema {nombre} ...")
    records, t0 = [], time.perf_counter()
    for reg in registros:
        consulta = reg["messages"][1]["content"]
        resultado = pipeline.answer_query(
            consulta, store, use_lora=USE_LORA, bm25=bm25,
            model_bundle=model_bundle, **banderas,
        )
        records.append(pipeline.to_eval_record(resultado, reg))
    latencias[nombre] = (time.perf_counter() - t0) / len(registros)
    corridas[nombre] = records
    print(f"  {nombre}: {latencias[nombre]:.2f} seg/consulta")

In [ ]:
# Persistir las tres corridas en Drive, para que el companero de evaluacion las
# levante sin re-generar (mismo patron que M1/M2 con sus salidas).
destino = pathlib.Path(config.DRIVE_ROOT) / "rag" / "corridas_abc"
destino.mkdir(parents=True, exist_ok=True)

for nombre, records in corridas.items():
    slug = nombre.split()[0].lower()   # A / B / C
    ruta = destino / f"eval_records_sistema_{slug}.json"
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    print(f"  {nombre} -> {ruta}")

# Latencia (unico numero que este notebook SI reporta: es la parte del costo que
# no necesita al juez). El delta de calidad lo llena la evaluacion.
print("\nLatencia por consulta:")
for nombre, seg in latencias.items():
    print(f"  {nombre}: {seg:.2f} s")

### Tabla de deltas -- esqueleto para el companero de evaluacion

La calidad (scorecard del harness + RAGAS) se llena corriendo la evaluacion
sobre los tres archivos `eval_records_sistema_{A,B,C}.json` de arriba. La
estructura de la tabla que va a la entrega M3:

| Sistema | Scorecard M2 | faithfulness | context precision | context recall | answer relevancy | seg/consulta |
|---|---|---|---|---|---|---|
| A ingenuo | _(evaluacion)_ | _(RAGAS)_ | _(RAGAS)_ | _(RAGAS)_ | _(RAGAS)_ | *(Fase 4)* |
| B +hybrid | | | | | | |
| C +rerank | | | | | | |

Como leerla (guia de S08):
- **B > A** en context precision/recall -> el corpus tiene vocabulario exacto que
  el denso difuminaba: el hybrid paga.
- **C > B** -> habia ruido en el top-k: el reranker lo limpia.
- **Algo no mejoro** -> no es fracaso, es diagnostico: esa tecnica no ataca EL
  fallo de Amparo. Reportarlo con la latencia -- "no valio su costo" es una
  conclusion de ingenieria valida.

## Fase 5 -- Tool use: el retrieval como herramienta (S10)

En vez de recuperar SIEMPRE, el modelo decide si llamar `buscar_normas`. La tool
invoca el retrieval avanzado (sistema C). Se muestran dos casos: uno que amerita
buscar (fundamentar en una norma) y uno que no (un saludo).

In [ ]:
from tools.rag import tools

for consulta in [
    "me despidieron sin justa causa, que derechos tengo",   # deberia buscar
    "hola, buenas tardes",                                    # no deberia buscar
]:
    print("=" * 78)
    print("USUARIO:", consulta)
    r = tools.responder_con_tools(consulta, store, bm25=bm25, model_bundle=model_bundle)
    for tc in r["tool_calls"]:
        print(f"  -> llamo {tc['tool']}({tc['args']})")
    print("AMPARO:", r["response"])

---
### Lo que se entrega

1. **Codigo** de las dos tecnicas (`tools/rag/hybrid.py`, `tools/rag/rerank.py`),
   integradas en `retrieve.py`/`pipeline.py` por bandera, y la tool
   (`tools/rag/tools.py`), todo con tests que corren sin GPU (`tests/rag/`).
2. **Las tres corridas A/B/C** en formato Ragas + la latencia por sistema
   (Fase 4), para que la evaluacion mida el delta.
3. **Las decisiones de diseno** en `docs/m3_decisiones_rag.md`.

*Amparo · M3 · RAG avanzado + tool use.*